# Teachable Machine, dataset propio y Gradio

En esta actividad vas a crear un clasificador de imágenes propio. La secuencia completa tiene cuatro momentos: diseñar el problema, recolectar imágenes, entrenar en Teachable Machine y probar el modelo desde una interfaz web con Gradio.

## Objetivos de aprendizaje

Al finalizar la actividad, vas a poder:

- definir clases visuales para un problema de clasificación;
- construir un dataset ordenado por carpetas;
- aplicar criterios de curaduría para reducir sesgos;
- entrenar y probar un modelo en Teachable Machine;
- exportar el modelo y usarlo desde Python con Gradio.

## 1. Diseñar el problema de clasificación

Antes de descargar imágenes, definí el problema. Un buen clasificador necesita clases claras y ejemplos representativos.

Completá estas decisiones:

- clases del modelo: entre 2 y 4 categorías;
- rasgos visuales esperados: qué debería mirar el modelo;
- posibles confusiones: qué clases podrían parecerse;
- contexto de uso: con qué tipo de imágenes se probará después.

**Consigna de trabajo:** escribí una hipótesis breve. Por ejemplo: “El modelo debería distinguir hojas de roble y hojas de eucalipto por la forma del borde y la nervadura, no por el fondo de la foto”.

## 2. Recolectar imágenes

### Método rápido: extensiones de navegador

Para trabajar en clase, las extensiones permiten armar carpetas en pocos minutos desde búsquedas de imágenes.

Opciones posibles:

- **Download All Images / ZipImageDownloader**: permite descargar muchas imágenes visibles en una pestaña.
- **Image Downloader**: muestra una grilla de imágenes y permite seleccionar cuáles descargar.

### Método in situ: webcam de Teachable Machine

Si el problema se puede resolver con objetos del aula, gestos o expresiones, la webcam de Teachable Machine permite capturar muchas imágenes rápidamente con variaciones de posición, luz y fondo.

## 3. Organizar carpetas

Teachable Machine puede cargar imágenes por clase si están ordenadas en carpetas.

Usá esta estructura:

```text
Dataset/
  clase_1/
    imagen_001.jpg
    imagen_002.jpg
  clase_2/
    imagen_001.jpg
    imagen_002.jpg
```

Los nombres de las carpetas deben representar las clases. Por ejemplo: `Dataset/gato` y `Dataset/perro`.

**Consigna de trabajo:** cada grupo debería reunir entre 50 y 100 imágenes por clase para una primera prueba.

## 4. Curar el dataset

El modelo aprende a partir de los ejemplos que recibe. Si los ejemplos están sesgados, el modelo puede aprender atajos incorrectos.

Revisá estos criterios:

- **Balance de clases**: cada clase debería tener una cantidad similar de imágenes.
- **Variedad**: incluí distintos fondos, ángulos, tamaños e iluminaciones.
- **Ruido**: eliminá imágenes borrosas, duplicadas o irrelevantes.
- **Sesgo de fondo**: evitá que una clase aparezca siempre con el mismo fondo.

Ejemplo: si todas las fotos de gatos están en un sillón rojo y todas las de perros están en pasto verde, el modelo puede aprender a distinguir fondos en lugar de animales.

**Consigna de lectura:** revisá 10 imágenes de cada clase. Marcá cuáles podrían introducir sesgo y explicá por qué.

Las imagenes que inducen sesgo son en principio las de fondos tipicos de cada clase. En el caso de Superman, fondo de cielo azul y luz de dia, en el caso de Batman, imagenes en la oscuridad, la noche.

## 5. Entrenar en Teachable Machine

Pasos sugeridos:

1. Entrá a Teachable Machine.
2. Creá un proyecto de imagen.
3. Cargá las carpetas de cada clase.
4. Entrená el modelo.
5. Probalo con webcam o con imágenes nuevas.

Hiperparámetros básicos:

- **Epochs**: cantidad de veces que el modelo recorre los ejemplos.
- **Batch Size**: cantidad de imágenes procesadas en cada paso.
- **Learning Rate**: tamaño de los ajustes durante el aprendizaje. Si no se modifica, conviene dejar el valor por defecto.

## 6. Hackear el modelo

Después de entrenar, intentá encontrar casos donde el modelo falle. Este paso es importante porque permite entender qué aprendió realmente.

Pruebas posibles:

- una imagen con fondo distinto;
- una imagen con poca luz;
- una imagen de un objeto parcialmente tapado;
- una imagen parecida a dos clases;
- una imagen que no pertenezca a ninguna clase.

**Consigna de análisis:** cuando el modelo falle, formulá una hipótesis. ¿Faltó variedad? ¿Había sesgo de fondo? ¿Las clases eran demasiado parecidas? ¿El dataset estaba desbalanceado?

El dataset esta balanceado, 500 imagenes de cada clase. Al abarcar historietas, series , dibujos animados y peliculas, cubre bastante bien todas las caracteristicas de cada clase.

## 7. Exportar el modelo

Para usar el modelo fuera de Teachable Machine:

1. Hacé clic en **Export Model**.
2. Elegí **TensorFlow / Keras**.
3. Descargá el archivo `.zip`.
4. Descomprimilo.
5. Ubicá estos archivos en la carpeta de trabajo:
   - `keras_model.h5`;
   - `labels.txt`.

## 8. Instalar dependencias para Gradio

Gradio permite crear una interfaz web simple para probar el modelo con imágenes nuevas.

In [ ]:
!pip install -q gradio tensorflow pillow numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 9. Cargar modelo y etiquetas

Esta celda espera que `keras_model.h5` y `labels.txt` estén en la misma carpeta del notebook.

In [ ]:
from pathlib import Path

import os
# IMPORTANTE: Esto debe ir ANTES de cualquier importación de tensorflow o keras
os.environ["TF_USE_LEGACY_KERAS"] = "1"

from pathlib import Path
import gradio as gr
import numpy as np
from PIL import Image, ImageOps

# Ahora importamos desde tf_backend o de la forma heredada
import tensorflow as tf
from tensorflow.keras.models import load_model

# Define CustomDepthwiseConv2D class to resolve NameError
# Assuming it's meant to be an alias or wrapper for tf.keras.layers.DepthwiseConv2D.
# If you have a specific custom implementation for this layer, replace this with your actual class.
class CustomDepthwiseConv2D(tf.keras.layers.DepthwiseConv2D):
    pass

# Updated paths to load from Google Drive
base_drive_path = Path('/content/drive/MyDrive/procesamiento digital de imagenes/004_Teacheable_Machine_Dataset_propio/converted_keras')
ruta_modelo = base_drive_path / "keras_model.h5"
ruta_etiquetas = base_drive_path / "labels.txt"

if not ruta_modelo.exists():
    raise FileNotFoundError(f"No se encontró el modelo en: {ruta_modelo}. Revisá que la ruta sea correcta y que Google Drive esté montado.")

if not ruta_etiquetas.exists():
    raise FileNotFoundError(f"No se encontraron las etiquetas en: {ruta_etiquetas}. Revisá que la ruta sea correcta y que Google Drive esté montado.")

# Define custom objects to handle potential version discrepancies
custom_objects = {
    'DepthwiseConv2D': CustomDepthwiseConv2D # Use our custom class here
}

# Use load_model for compatibility with Keras 2 models in a Keras 3 environment (with TF_USE_LEGACY_KERAS=1)
modelo_teachable = load_model(ruta_modelo, compile=False, custom_objects=custom_objects)

with ruta_etiquetas.open("r", encoding="utf-8") as archivo_etiquetas:
    lineas_etiquetas = archivo_etiquetas.readlines()

nombres_clases = []

for linea in lineas_etiquetas:
    nombre_clase = linea.strip().split(" ", 1)[-1]
    nombres_clases.append(nombre_clase)

print("Clases cargadas:", nombres_clases)

Clases cargadas: ['Batman', 'Superman']


## 10. Definir la función de predicción

Teachable Machine espera imágenes de 224 x 224 píxeles y valores normalizados al rango `[-1, 1]`. Por eso adaptamos cada imagen antes de enviarla al modelo.

In [ ]:
def preprocesar_imagen_para_teachable(imagen_numpy):
    imagen_pil = Image.fromarray(imagen_numpy.astype("uint8"), "RGB")

    tamano_esperado = (224, 224)
    imagen_redimensionada = ImageOps.fit(
        imagen_pil,
        tamano_esperado,
        Image.Resampling.LANCZOS,
    )

    arreglo_imagen = np.asarray(imagen_redimensionada)
    arreglo_imagen = arreglo_imagen.astype(np.float32)
    arreglo_imagen = (arreglo_imagen / 127.5) - 1

    lote_imagenes = np.expand_dims(arreglo_imagen, axis=0)

    return lote_imagenes


def predecir_imagen(imagen_entrada):
    if imagen_entrada is None:
        return {"Sin imagen": 1.0}

    lote_imagenes = preprocesar_imagen_para_teachable(imagen_entrada)
    prediccion = modelo_teachable.predict(lote_imagenes, verbose=0)

    probabilidades = {}

    for indice, nombre_clase in enumerate(nombres_clases):
        puntaje = float(prediccion[0][indice])
        probabilidades[nombre_clase] = puntaje

    return probabilidades

**Consigna de lectura:** identificá dónde se redimensiona la imagen y dónde se normaliza. ¿Por qué esos pasos deben coincidir con el entrenamiento original?

tamano_esperado = (224, 224)

imagen_redimensionada = ImageOps.fit(
    imagen_pil,
    tamano_esperado,
    Image.Resampling.LANCZOS,
)

aqui corta y ajusta la imagen original para que mida exactamente 224x224 píxeles, sin deformarla, usando un filtro de alta calidad (LANCZOS).



## 11. Crear la interfaz con Gradio

La interfaz permite subir una imagen o usar la cámara, según el entorno donde ejecutes el notebook. El parámetro `share=True` genera un enlace temporal para probar el modelo desde otros dispositivos.

In [ ]:
demo = gr.Interface(
    fn=predecir_imagen,
    inputs=gr.Image(type="numpy"),
    outputs=gr.Label(num_top_classes=3),
    title="Clasificador de imágenes con Teachable Machine",
    description="Subí una imagen para probar el modelo entrenado en clase.",
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b9e7191f21fe8c8a84.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 12. Cierre y discusión

Para cerrar la actividad, compará el resultado del modelo con las decisiones tomadas durante la recolección de imágenes.

Preguntas para discutir:

- ¿Qué clases funcionaron mejor?
La clase superman tiene mejor precision aunque la diferencia es muy poca. Esto es porque el patron de colores es muy determinante, caracteristico. brillante y constante (el traje azul, la capa roja y la "S" gigante en el pecho). Batman cambia mucho según la versión. A veces negro , a veces de gris, a veces con armadura o su traje es de cómics antiguos (azul y gris), lo que confunde más al modelo.
- ¿Qué errores se repitieron?
Confusión con la oscuridad/sombras: Muchas imágenes de Superman en escenas oscuras o lluviosas el modelo las clasifica erróneamente como Batman solo por la paleta de colores sombría.
- ¿Qué imágenes faltaron en el dataset original?
Imágenes de contexto variado. Más fotos de Batman de día o en fondos claros, y más fotos de Superman de noche. El problema es cuando hay imagenes donde aparecen ambos ( cosa que se da en las peliculas ) el modelo se confunde. Pareciera que elige al azar.
- ¿El modelo aprendió el objeto principal o algún rasgo del fondo?
 Es muy probable que el modelo haya aprendido el fondo y los colores generales antes que al personaje en sí. Entonces influye el medio ( la pelicula, la historieta, la serie, la animacion, el muñeco y el año ) Si el fondo es ciudad con edificios y noche, clasifica como "Batman" Si el fondo es un cielo azul brillante o ciudad de día, clasifica como "Superman". Aprendió que Oscuridad = Batman y Claridad = Superman.
- ¿Qué cambiarías si tuvieras que entrenarlo otra vez?
Agregar las clases "Ambos" y "ninguno" y ampliar la diversidad de estilos. Asegurar imágenes de películas reales (fotogramas de videos) y de cómics/animación para ambas clases, logrando que el modelo entienda el concepto abstracto de "Batman" y "Superman" (la máscara con orejas, la capa negra, la capa roja, el murcielago, la S) y no solo la foto de un actor específico.



hay una carpeta mas, llamada testeo, con imagenes dificiles. El modelo se desempeño bien incluso con estas imagenes.